# Lab 06, part 1: CSV vs JSON vs Parquet vs ORC

The same 500,000 rows are written four ways. Then we compare size on HDFS and the time to read one column.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("lab-06-file-formats")
    .config("spark.hadoop.dfs.replication", "1")
    .getOrCreate()
)
BASE = "hdfs://namenode:8020/user/lab/06/formats"

def hdfs_summary(path):
    """Return (directories, files, bytes) under an HDFS path, like `hdfs dfs -count`."""
    jvm = spark._jvm
    p = jvm.org.apache.hadoop.fs.Path(path)
    fs = p.getFileSystem(spark._jsc.hadoopConfiguration())
    s = fs.getContentSummary(p)
    return s.getDirectoryCount(), s.getFileCount(), s.getLength()

## 1. Generate the rows

`spark.range` makes the ids. The other columns come from `rand` with fixed seeds, so every run gives the same data.

In [ ]:
countries = ["Nepal", "India", "Japan", "Brazil", "Kenya"]

df = spark.range(500_000).select(
    F.col("id").alias("order_id"),
    (F.rand(1) * 10_000).cast("int").alias("user_id"),
    F.element_at(F.array(*[F.lit(c) for c in countries]), (F.col("id") % 5 + 1).cast("int")).alias("country"),
    F.round(F.rand(2) * 500, 2).alias("amount"),
    F.concat(F.lit("order note number "), (F.col("id") % 1000).cast("string")).alias("comment"),
    F.expr("timestamp_seconds(1767225600 + id * 7)").alias("ts"),
)
df.printSchema()
df.show(3, truncate=False)

## 2. Write four copies

`coalesce(2)` keeps the file count the same for every format, so the comparison is fair.

In [ ]:
small = df.coalesce(2)
small.write.mode("overwrite").option("header", True).csv(f"{BASE}/csv")
small.write.mode("overwrite").json(f"{BASE}/json")
small.write.mode("overwrite").parquet(f"{BASE}/parquet")
small.write.mode("overwrite").orc(f"{BASE}/orc")
print("done")

## 3. Compare sizes

Same numbers as `hdfs dfs -du -h /user/lab/06/formats` in a terminal.

In [ ]:
import pandas as pd

sizes = pd.DataFrame(
    [(fmt, hdfs_summary(f"{BASE}/{fmt}")[2] / 1024 / 1024) for fmt in ["csv", "json", "parquet", "orc"]],
    columns=["format", "size_mb"],
)
sizes

## 4. Compare read speed for one column

We compute the average of `amount`. A row format must parse every column of every row. Parquet and ORC can read just the `amount` column.

Each read runs three times and we keep the best time, so the first-run warm-up does not distort it. Your numbers will differ from the README; the order should not.

In [ ]:
import time

schema = "order_id LONG, user_id INT, country STRING, amount DOUBLE, comment STRING, ts TIMESTAMP"
readers = {
    "csv": lambda: spark.read.schema(schema).option("header", True).csv(f"{BASE}/csv"),
    "json": lambda: spark.read.schema(schema).json(f"{BASE}/json"),
    "parquet": lambda: spark.read.parquet(f"{BASE}/parquet"),
    "orc": lambda: spark.read.orc(f"{BASE}/orc"),
}

rows = []
for fmt, read in readers.items():
    best = min(
        (lambda t0: (read().agg(F.avg("amount")).collect(), time.perf_counter() - t0)[1])(time.perf_counter())
        for _ in range(3)
    )
    rows.append((fmt, best))

speed = pd.DataFrame(rows, columns=["format", "seconds"])
speed

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, (left, right) = plt.subplots(1, 2, figsize=(10, 3.5))
sns.barplot(data=sizes, x="format", y="size_mb", color="#f5b301", ax=left)
left.set_title("Size on HDFS (MB)")
sns.barplot(data=speed, x="format", y="seconds", color="#e8590c", ax=right)
right.set_title("Read one column (seconds, best of 3)")
plt.tight_layout()
plt.show()

## 5. See the file layout

Spark writes one file per partition plus an empty `_SUCCESS` marker. Each file name tells you the format.

In [ ]:
for fmt in ["csv", "json", "parquet", "orc"]:
    dirs, files, size = hdfs_summary(f"{BASE}/{fmt}")
    print(f"{fmt:8} files={files} bytes={size}")

In [ ]:
spark.stop()